<a href="https://colab.research.google.com/github/KarimBekhtiGalvao/Diffusion-Based-Generation/blob/main/Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Data Preparation

from google.colab import drive
drive.mount('/content/drive')
data_dir = "/content/drive/MyDrive/Cours/CAI_projet"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

for name in os.listdir(data_dir):
    print(name)

raw_Pride_and_Prejudice.txt
raw_Sense_and_Sensibility.txt
raw_Persuasion.txt
raw_Emma.txt
raw_Northanger_Abbey.txt


In [ ]:
texts = ""

for name in os.listdir(data_dir):
    if name.endswith(".txt"):
        path = os.path.join(data_dir, name)
        with open(path, "r", encoding="utf-8") as f:
            texts += f.read() + "\n"
texts[:100]

In [ ]:
import re
import unicodedata

import re
import unicodedata

def normalize_text(text: str) -> str:
    # Remove BOM
    text = text.replace("\ufeff", "")

    # Turn literal \n into real newlines
    text = text.replace("\\n", "\n")

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Lowercase
    text = text.lower()

    # Normalize quotes/dashes
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("—", "-").replace("–", "-")

    # Remove Gutenberg artifacts
    text = re.sub(r"\[illustration\]", "", text, flags=re.IGNORECASE)
    text = text.replace("_", "")

    # Replace newlines with spaces
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\n", " ")

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

texts = normalize_text(texts)
texts[:100]

'preface. walt whitman has somewhere a fine and just distinction between "loving by allowance" and "l'

In [ ]:
from collections import Counter
### [EXTRACT OF CODE ADAPTED FROM LAB 5]

# Top N words
N = 25

# Convert a list of sentences into a single big string
all_text = "".join([i for i in texts])# Your code here. Aim for 1 line.

# Split the text into a list of words
words = all_text.split()# Your code here. Aim for 1 line.

# Count frequency
word_counts = Counter(words)# Your code here. Aim for 1 line.

# Take the top N most frequent words
top_words = [w for w, c in word_counts.most_common(N)]
print(top_words)

['the', 'to', 'of', 'and', 'a', 'her', 'in', 'was', 'i', 'she', 'not', 'be', 'that', 'it', 'had', 'he', 'as', 'for', 'you', 'his', 'with', 'have', 'but', 'at', 'is']


The top 25 words used in english according to Wikipedia are:
the, be, to, of, and, a, in, that, have, I, it, for, not, on, with, he, as, you, do, at, this, but, his, by, from, meaning most of the common words in the dataset are the most common words in english: it is a somewhat representative subset.

In [ ]:
### [EXTRACT OF CODE ADAPTED FROM LAB 5]

class CharBPETokenizer:
    def __init__(self, vocab_size=100):

        # Maximum vocabulary size for BPE
        self.vocab_size = vocab_size

        # Set to store current vocabulary
        self.vocab = set()

        # Special tokens used by BERT
        self.special_tokens = ["[UNK]", "[MASK]", "[SEP]"]

    def train(self, text):
        """
        Learn Byte-Pair Encoding (BPE) merges from the input text.
        BPE merges frequent consecutive character pairs into new tokens
        until the vocabulary reaches the desired size.
        """

        # Convert text to list of characters
        tokens = [c for word in text for c in word] # Your code here. Aim for 1 line

        # Initialize the vocabulary with the characters
        vocab = Counter(tokens)

        # Iteratively merge frequent pairs until vocab_size is reached
        while len(vocab) < self.vocab_size:

            pair_counts = Counter()

            # Count how many times each consecutive pair of tokens occurs in the text
            # The result, pair_counts, is a dictionary mapping each pair to its frequency
            # Your code here. Aim for 3-4 lines. 1
            for i in range(len(tokens)-1):
              pair = (tokens[i], tokens[i+1])
              pair_counts[pair] += 1

            if not pair_counts:
                break  # No more pairs to merge

            # Find the most frequent pair
            #print(pair_counts.most_common()) # [(('a', 'a'), 4), (('a', 'b'), 2), (('b', 'd'), 1), (('d', 'a'), 1), (('b', 'a'), 1), (('a', 'c'), 1)]
            best_pair, freq = pair_counts.most_common(1)[0]# Your code here. Use the .most_common method.

            # Merge the most frequent pair into a new token
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best_pair:
                    new_tokens.append(tokens[i] + tokens[i + 1]) # Append new token # Your code here. Aim for 1 line
                    i += 2# Skip merged token  # Your code here. Aim for 1 line
                else:
                    new_tokens.append(tokens[i]) # Append old token
                    i += 1# Go to the next token # Your code here. Aim for 1 line

            tokens = new_tokens
            vocab = Counter(tokens)


        # Combine learned tokens, original characters, and special tokens
        vocab_set = set(tokens) | set(text) | set(self.special_tokens)

        # Sort vocab_set by length (longest first)
        self.vocab_list = sorted(vocab_set, key=len, reverse=True) # Your code here. Aim for 1 line.

        # Mapping from token to ID with a vocabulary
        self.token_to_id = {token: i for i, token in enumerate(self.vocab_list)}# Your code here. Aim for a line.

        # Mapping from Id to tokens with a vocabulary
        self.id_to_token = {i: token for i, token in enumerate(self.vocab_list)}# Your code here. Aim for a line.


    def tokenize(self, text):
        """
        Tokenize input text. We replace longest tokens first.
        If a character sequence is not in the vocabulary, replace it with [UNK].
        """
        tokens = []
        i = 0

        while i < len(text):
            match_found = False

            # Try to match the longest token first
            for token in self.vocab_list:
                if text.startswith(token, i):
                    tokens.append(token)
                    i += len(token)# Go to the next token. Your code here. Aim for 1 line.
                    match_found = True
                    break
            if not match_found:
                tokens.append("[UNK]")  # Replace unknown character
                print(f"[WARNING] Character '{text[i]}' not in vocabulary, replaced with [UNK]")
                i += 1# Go to the next token. Your code here. Aim for 1 line.

        return tokens

    def encode_ids(self, text):
        """
        Convert a text string into a list of token IDs
        """
        # Convert raw text into tokens
        tokens = self.tokenize(text)# Your code here. Aim for 1 line.

        # Assing index to each token
        token_ids = [self.token_to_id[t] for t in tokens]# Your code here. Aim for 1 line.
        return token_ids

    def detokenize(self, tokens):
        """
        Convert a list of tokens back into a text string
        """
        return "".join(tokens)

    def decode_ids(self, ids):
        """
        Convert a list of token IDs back into text
        """
        # Convert ids to tokens
        tokens = [self.id_to_token[i] for i in ids] # Your code here. Aim for 1 line

        # Conver tokens into text (detokenize)
        text =  "".join(tokens) # Your code here. Aim for 1 line
        return text

vocab_size = 100
tokenizer = CharBPETokenizer(vocab_size=vocab_size)

# Prepare the text for training the tokenizer
tokenizer_corpus = "".join(texts)
tokenizer_corpus = tokenizer_corpus.replace("\n", " ")

# Train the tokenizer
tokenizer.train(tokenizer_corpus)
